# FEDformer — Multi-seed NVDA con CP Walk-forward

Ejecuta 4 seeds (42, 123, 7, 456) con la config canónica NVDA + `--cp-walkforward`.

**Objetivo:** determinar si el gap de cobertura CP en fold 2 (seed=42: coverage=45.8%) es
seed-específico (varianza de inicialización NF) o sistemático (régimen de datos).

**Config canónica:** `seq_len=96 · pred_len=20 · batch_size=64 · splits=4 · log_return · clip=0.5`

**Tiempo estimado en T4 (40 SMs):** ~4-6 min/seed × 4 seeds ≈ 20-25 min total.

> **Nota T4:** torch.compile se activa (40 SMs ≥ umbral). Si aparecen NaN en loss, añadir
> `"--preset", "cpu_safe"` a `EXTRA_ARGS` abajo.

In [ ]:
# ── Celda 1: Verificar GPU ────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    sms = props.multi_processor_count
    print(f"GPU : {props.name}")
    print(f"SMs : {sms}")
    print(f"VRAM: {props.total_memory // 1024**3} GB")
    if sms >= 40:
        print("torch.compile ACTIVO (SMs >= 40) — si hay NaN añadir --preset cpu_safe")
    else:
        print(f"torch.compile DESACTIVADO (SMs={sms} < 40) — degrada a eager")
else:
    print("⚠ No hay GPU disponible — los runs serán lentos en CPU")

In [ ]:
# ── Celda 2: Instalar dependencias ────────────────────────────────────────
# torch, pandas, numpy, sklearn, scipy, matplotlib ya están en Kaggle
!pip install -q optuna pandas-ta-classic

In [ ]:
# ── Celda 3: Clonar repositorio ───────────────────────────────────────────
import os
import sys

REPO = "https://github.com/RubenPanero/FEDformer-Probabilistic-Time-Series-Forecasting.git"
WORKDIR = "/kaggle/working/fedformer"

if not os.path.exists(WORKDIR):
    !git clone {REPO} {WORKDIR}
else:
    print("Repo ya clonado — actualizando")
    !git -C {WORKDIR} pull

os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

print(f"Directorio: {os.getcwd()}")
!git log --oneline -3

In [ ]:
# ── Celda 4: Verificar dataset NVDA ───────────────────────────────────────
import pandas as pd

df = pd.read_csv("data/NVDA_features.csv")
print(f"NVDA dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"Rango fechas: {df.iloc[0, 0]} → {df.iloc[-1, 0]}")
print(f"Columnas: {df.columns.tolist()}")

In [ ]:
# ── Celda 5: Configuración ────────────────────────────────────────────────
import subprocess
import time

SEEDS = [42, 123, 7, 456]

EXTRA_ARGS = [
    "--seq-len", "96",
    "--pred-len", "20",
    "--batch-size", "64",
    "--splits", "4",
    "--return-transform", "log_return",
    "--metric-space", "returns",
    "--gradient-clip-norm", "0.5",
    "--cp-walkforward",
    # "--preset", "cpu_safe",  # descomentar si hay NaN en loss
]

ENV = {**os.environ, "MPLBACKEND": "Agg"}
RESULTS = []

def run_seed(seed: int) -> dict:
    """Ejecuta main.py con un seed y devuelve {seed, returncode, elapsed_min}."""
    cmd = [
        sys.executable, "main.py",
        "--csv", "data/NVDA_features.csv",
        "--targets", "Close",
        "--seed", str(seed),
        "--save-results",
        "--no-show",
    ] + EXTRA_ARGS
    print(f"\n{'='*60}")
    print(f"  Seed {seed} — iniciando")
    print(f"{'='*60}")
    t0 = time.time()
    result = subprocess.run(cmd, env=ENV)
    elapsed = (time.time() - t0) / 60
    ok = result.returncode == 0
    status = "✓ OK" if ok else f"✗ ERROR (code={result.returncode})"
    print(f"\nSeed {seed}: {status} — {elapsed:.1f} min")
    return {"seed": seed, "returncode": result.returncode, "elapsed_min": round(elapsed, 1)}

print(f"Seeds planificados: {SEEDS}")
print(f"Args extra: {' '.join(EXTRA_ARGS)}")

In [ ]:
# ── Celda 6: Seed 42 ──────────────────────────────────────────────────────
RESULTS.append(run_seed(42))

In [ ]:
# ── Celda 7: Seed 123 ─────────────────────────────────────────────────────
RESULTS.append(run_seed(123))

In [ ]:
# ── Celda 8: Seed 7 ───────────────────────────────────────────────────────
RESULTS.append(run_seed(7))

In [ ]:
# ── Celda 9: Seed 456 ─────────────────────────────────────────────────────
RESULTS.append(run_seed(456))

In [ ]:
# ── Celda 10: Agregar métricas de los manifiestos ─────────────────────────
import json
import numpy as np
from pathlib import Path

results_dir = Path("results")
manifests = sorted(results_dir.glob("run_manifest_*_NVDA_features.json"),
                   key=lambda p: p.stat().st_mtime)
# Tomar los últimos len(SEEDS) manifiestos (los de este experimento)
manifests = manifests[-len(SEEDS):]

rows = []
for mpath in manifests:
    m = json.loads(mpath.read_text())
    metrics = m.get("metrics", {})
    seed_val = m.get("seed", m.get("config", {}).get("seed", "?"))
    ts = m.get("timestamp", mpath.stem.split("_")[2] + "_" + mpath.stem.split("_")[3])
    rows.append({
        "seed":                 seed_val,
        "timestamp":            ts,
        "sharpe":               metrics.get("sharpe_ratio", float("nan")),
        "sortino":              metrics.get("sortino_ratio", float("nan")),
        "max_drawdown":         metrics.get("max_drawdown", float("nan")),
        "var_95":               metrics.get("var_95", float("nan")),
        "cp_wf_coverage_80":    metrics.get("cp_wf_coverage_80", float("nan")),
        "cp_wf_folds_calibrated": metrics.get("cp_wf_folds_calibrated", 0),
        "pinball_p50":          metrics.get("pinball_p50", float("nan")),
        "coverage_80":          metrics.get("coverage_80", float("nan")),
    })

agg = pd.DataFrame(rows).sort_values("seed")

print("=" * 65)
print("  MÉTRICAS POR SEED")
print("=" * 65)
print(agg[["seed", "sharpe", "sortino", "cp_wf_coverage_80",
           "cp_wf_folds_calibrated", "pinball_p50"]].to_string(index=False))

cp_vals = agg["cp_wf_coverage_80"].dropna()
sharpe_vals = agg["sharpe"].dropna()
print(f"\n{'='*65}")
print(f"  RESUMEN cp_wf_coverage_80:  mean={cp_vals.mean():.4f}  std={cp_vals.std():.4f}")
print(f"  Objetivo ≥ 0.80:  {'CUMPLIDO' if cp_vals.mean() >= 0.80 else 'NO cumplido'}  "
      f"({(cp_vals >= 0.80).sum()}/{len(cp_vals)} seeds)")
print(f"  RESUMEN sharpe:             mean={sharpe_vals.mean():.4f}  std={sharpe_vals.std():.4f}")

In [ ]:
# ── Celda 11: Análisis por fold — ¿es fold 2 seed-específico? ────────────
from utils.calibration import conformal_calibration_walkforward, apply_conformal_interval

def fold_coverage(predictions_csv: Path, alpha: float = 0.2) -> dict:
    """Calcula cobertura y q_hat por fold para un CSV de predicciones."""
    df = pd.read_csv(predictions_csv)
    fold_data = {
        int(f): np.abs(sub["pred"].values - sub["ground_truth"].values)
        for f, sub in df.groupby("fold")
    }
    q_hat_by_fold = conformal_calibration_walkforward(fold_data, alpha=alpha)
    result = {}
    for fold, q in q_hat_by_fold.items():
        if q is None:
            result[fold] = {"q_hat": None, "coverage": None,
                            "err_mean": fold_data[fold].mean(),
                            "err_p80": np.percentile(fold_data[fold], 80)}
            continue
        sub = df[df["fold"] == fold]
        pred, gt = sub["pred"].values, sub["ground_truth"].values
        lo, hi = pred - q, pred + q
        covered = (gt >= lo) & (gt <= hi)
        result[fold] = {
            "q_hat":    round(float(q), 5),
            "coverage": round(float(covered.mean()), 4),
            "err_mean": round(float(fold_data[fold].mean()), 5),
            "err_p80":  round(float(np.percentile(fold_data[fold], 80)), 5),
        }
    return result

# Emparejar cada seed con su predictions CSV via timestamp del manifest
pred_csvs = sorted(results_dir.glob("predictions_*_NVDA_features.csv"),
                   key=lambda p: p.stat().st_mtime)[-len(SEEDS):]

fold_results = {}  # seed → {fold → metrics}
for mpath, ppath in zip(manifests, pred_csvs):
    m = json.loads(mpath.read_text())
    seed_val = m.get("seed", "?")
    fold_results[seed_val] = fold_coverage(ppath)

# Tabla por fold por seed
print("=" * 75)
print("  COBERTURA POR FOLD POR SEED (alpha=0.2, target≥0.80)")
print("=" * 75)

all_folds = sorted({f for fc in fold_results.values() for f in fc})
header = f"{'Fold':<6}" + "".join(f"{'Seed '+str(s):<15}" for s in sorted(fold_results))
print(header)
print("-" * 75)

for fold in all_folds:
    row = f"{fold:<6}"
    for seed in sorted(fold_results):
        info = fold_results[seed].get(fold, {})
        cov = info.get("coverage")
        if cov is None:
            row += f"{'(calibración)':<15}"
        else:
            flag = " ✓" if cov >= 0.80 else " ✗"
            row += f"{cov:.4f}{flag:<9}"
    print(row)

print()
print("=" * 75)
print("  ERROR DEL MODELO (err_mean) POR FOLD POR SEED")
print("=" * 75)
print(header)
print("-" * 75)
for fold in all_folds:
    row = f"{fold:<6}"
    for seed in sorted(fold_results):
        info = fold_results[seed].get(fold, {})
        em = info.get("err_mean")
        row += f"{em:<15.5f}" if em is not None else f"{'?':<15}"
    print(row)

print()
# ¿Fold 2 es sistemáticamente malo?
fold2_coverages = [
    fold_results[s].get(2, {}).get("coverage")
    for s in fold_results
    if fold_results[s].get(2, {}).get("coverage") is not None
]
fold2_errs = [
    fold_results[s].get(2, {}).get("err_mean")
    for s in fold_results
    if fold_results[s].get(2, {}).get("err_mean") is not None
]
if fold2_coverages:
    print(f"Fold 2 coverage:  {fold2_coverages}")
    print(f"  mean={np.mean(fold2_coverages):.4f}  std={np.std(fold2_coverages):.4f}")
    print(f"  {'SISTEMÁTICO (todos los seeds < 0.80)' if all(c < 0.80 for c in fold2_coverages) else 'SEED-ESPECÍFICO (algunos seeds ok)'}")
    print(f"Fold 2 err_mean:  mean={np.mean(fold2_errs):.5f}  (ref fold1 seed42=0.0280)")

In [ ]:
# ── Celda 12: Guardar resumen JSON ────────────────────────────────────────
import json

summary = {
    "experiment": "multiseed_nvda_cp_walkforward",
    "seeds": SEEDS,
    "config": {
        "seq_len": 96, "pred_len": 20, "batch_size": 64, "splits": 4,
        "return_transform": "log_return", "metric_space": "returns",
        "gradient_clip_norm": 0.5, "alpha": 0.2,
    },
    "per_seed": agg.to_dict(orient="records"),
    "aggregate": {
        "cp_wf_coverage_80_mean": round(float(cp_vals.mean()), 4),
        "cp_wf_coverage_80_std":  round(float(cp_vals.std()), 4),
        "cp_wf_seeds_above_08":   int((cp_vals >= 0.80).sum()),
        "sharpe_mean":            round(float(sharpe_vals.mean()), 4),
        "sharpe_std":             round(float(sharpe_vals.std()), 4),
    },
    "per_fold": {
        str(s): {str(f): v for f, v in fc.items()}
        for s, fc in fold_results.items()
    },
    "fold2_diagnosis": {
        "coverages": fold2_coverages,
        "is_systematic": all(c < 0.80 for c in fold2_coverages) if fold2_coverages else None,
        "err_mean_mean": round(float(np.mean(fold2_errs)), 5) if fold2_errs else None,
    },
    "run_results": RESULTS,
}

out_path = "/kaggle/working/multiseed_nvda_cp_summary.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Resumen guardado: {out_path}")
print()
print("=" * 65)
print("  VEREDICTO FINAL")
print("=" * 65)
print(f"  cp_wf_coverage_80:  mean={summary['aggregate']['cp_wf_coverage_80_mean']:.4f}  "
      f"std={summary['aggregate']['cp_wf_coverage_80_std']:.4f}")
print(f"  Seeds con cov ≥ 0.80: {summary['aggregate']['cp_wf_seeds_above_08']}/{len(SEEDS)}")
print(f"  Sharpe:             mean={summary['aggregate']['sharpe_mean']:.4f}  "
      f"std={summary['aggregate']['sharpe_std']:.4f}")
diag = summary['fold2_diagnosis']
if diag['is_systematic'] is not None:
    tipo = 'SISTEMÁTICO (régimen de datos)' if diag['is_systematic'] else 'SEED-ESPECÍFICO (varianza NF)'
    print(f"  Gap fold 2: {tipo}")
print()
print(json.dumps(summary["aggregate"], indent=2))